# Time-Series Datasets in `generatedata`

This notebook documents and demonstrates the **time-series / sequence** layer added to the library.  
Three components work together:

| Component | Location | What it does |
|---|---|---|
| `save_data(..., seq_len, step_size)` | `generatedata/save_data.py` | Stores sequence metadata alongside the flat parquet files |
| `generate_mnist_sequence()` / `generate_mnist1d_sequence()` | `generatedata/data_generators.py` | Generates MNIST-family datasets with sequence metadata |
| `load_data_as_sequence()` | `generatedata/load_data.py` | Loads the flat data and reshapes it into `(num_points, seq_len, step_size)` |

The flat parquet storage format is **unchanged** — sequence metadata is purely descriptive and lives in `info.json`.

In [30]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display

from generatedata.load_data import load_data, load_data_as_sequence, data_names

---
## 1  API Reference

### 1.1  `save_data()` — new parameters

```python
save_data(
    data_dir,
    name: str,
    start_data: dict,
    target_data: dict,
    x_y_index: int | None = None,
    onehot_y: bool = False,
    additional_info: dict | None = None,
    seq_len: int | None = None,   # ← NEW: number of timesteps
    step_size: int | None = None, # ← NEW: features per timestep
) -> None
```

When both `seq_len` and `step_size` are provided:
- Validates `seq_len * step_size == x_y_index` (raises `ValueError` otherwise)
- Writes `seq_len` and `step_size` into `info.json` for downstream loaders

---

### 1.2  `generate_mnist_sequence()` / `generate_mnist1d_sequence()`

```python
generate_mnist_sequence(
    data_dir: Path,
    num_points: int = 1000,
    pixels_per_step: int = 1,        # step_size
    dataset_name: str = "MNIST",     # or 'EMNIST', 'KMNIST', 'FashionMNIST'
) -> None

generate_mnist1d_sequence(
    data_dir: Path,
    num_points: int = 1000,
    pixels_per_step: int = 1,
) -> None
```

**Naming convention** — the generated dataset name encodes the variant:

| Call | Dataset name |
|---|---|
| `generate_mnist_sequence(pixels_per_step=1)` | `MNIST_seq1` |
| `generate_mnist_sequence(pixels_per_step=28, dataset_name='FashionMNIST')` | `FashionMNIST_seq28` |
| `generate_mnist1d_sequence(pixels_per_step=1)` | `MNIST1D_seq1` |
| `generate_mnist1d_sequence(pixels_per_step=5)` | `MNIST1D_seq5` |

---

### 1.3  `load_data_as_sequence()` — the new load function

```python
load_data_as_sequence(
    name: str,
    local: bool = False,
    data_dir: Path | str | None = None,
    label_every_step: bool = True,
) -> tuple[np.ndarray, np.ndarray]
```

**Parameters**

| Parameter | Description |
|---|---|
| `name` | Dataset name (must have `seq_len` / `step_size` in metadata) |
| `local` | Load from the local `data/processed/` directory |
| `data_dir` | Override the default data directory path |
| `label_every_step` | If `True`, broadcast the label vector to every timestep and concatenate with the pixel sequence |

**Return shapes**

| `label_every_step` | `X_seq.shape` | `labels.shape` |
|---|---|---|
| `True` (default) | `(num_points, seq_len, step_size + label_dim)` | `(num_points, label_dim)` |
| `False` | `(num_points, seq_len, step_size)` | `(num_points, label_dim)` |

Both modes always return a consistent `(X_seq, labels)` two-tuple.  
Calling on a dataset **without** sequence metadata raises a helpful `ValueError`.

---
## 2  Discovering available sequence datasets

In [31]:
all_names = data_names(local=True)
seq_names = [n for n in all_names if '_seq' in n]

print(f"All available datasets ({len(all_names)}):")
for n in sorted(all_names):
    tag = "  ← sequence" if '_seq' in n else ""
    print(f"  {n}{tag}")

All available datasets (14):
  EMlocalization
  LunarLander
  MNIST
  MNIST1D
  MNIST1D_seq1  ← sequence
  MNIST1Dcustom_scale0.4_maxtrans48_corrnoise0.25_iidnoise0.02_shear0.75
  MNIST_custom_degrees0_0_translate0_0_scale1_1
  MNIST_seq1  ← sequence
  MassSpec
  circle
  manifold
  pca_line
  regression_circle
  regression_line


---
## 3  Inspecting sequence metadata

The `info` dict returned by `load_data()` carries the sequence parameters written by `save_data()`.

In [32]:
for name in seq_names:
    info = load_data(name, local=True)["info"]
    print(f"\n=== {name} ===")
    for k, v in info.items():
        print(f"  {k:15s}: {v}")


=== MNIST1D_seq1 ===
  num_points     : 1000
  size           : 50
  x_y_index      : 40
  x_size         : 40
  y_size         : 10
  onehot_y       : 1
  seq_len        : 40
  step_size      : 1
  data_family    : MNIST1D

=== MNIST_seq1 ===
  num_points     : 1000
  size           : 794
  x_y_index      : 784
  x_size         : 784
  y_size         : 10
  onehot_y       : 1
  seq_len        : 784
  step_size      : 1
  data_family    : MNIST


---
## 4  Basic loading — shapes and dtypes

In [33]:
# Choose a dataset to explore in this section
DATASET = "MNIST_seq1"

# --- label_every_step=True (default) ---
X_true, labels_true = load_data_as_sequence(DATASET, local=True, label_every_step=True)
print("label_every_step=True")
print(f"  X_seq  shape : {X_true.shape}   dtype: {X_true.dtype}")
print(f"  labels shape : {labels_true.shape}   dtype: {labels_true.dtype}")

print()

# --- label_every_step=False ---
X_false, labels_false = load_data_as_sequence(DATASET, local=True, label_every_step=False)
print("label_every_step=False")
print(f"  X_seq  shape : {X_false.shape}   dtype: {X_false.dtype}")
print(f"  labels shape : {labels_false.shape}   dtype: {labels_false.dtype}")

label_every_step=True
  X_seq  shape : (1000, 784, 11)   dtype: float32
  labels shape : (1000, 10)   dtype: float32

label_every_step=False
  X_seq  shape : (1000, 784, 1)   dtype: float32
  labels shape : (1000, 10)   dtype: float32


### How the label is embedded (label_every_step=True)

When `label_every_step=True` the label vector is broadcast to every timestep and appended as the
last `label_dim` channels of `X_seq`.  This makes it easy to pass the sequence directly to an RNN
or Transformer without a separate label input.

```
X_seq[:, t, :step_size]   → pixel values at timestep t
X_seq[:, t, step_size:]   → one-hot label (same for all t)
```

In [34]:
info = load_data(DATASET, local=True)["info"]
step_size = info["step_size"]
label_dim = info["y_size"]

sample_idx = 0
timestep   = 0

print(f"Sample {sample_idx}, timestep {timestep}:")
print(f"  pixel channel(s) : {X_true[sample_idx, timestep, :step_size]}")
print(f"  label channel(s) : {X_true[sample_idx, timestep, step_size:]}")
print(f"  standalone label : {labels_true[sample_idx]}")
print(f"  labels match?    : {np.allclose(X_true[sample_idx, timestep, step_size:], labels_true[sample_idx])}")

Sample 0, timestep 0:
  pixel channel(s) : [-1.]
  label channel(s) : [0. 0. 0. 0. 0. 0. 0. 1. 0. 0.]
  standalone label : [0. 0. 0. 0. 0. 0. 0. 1. 0. 0.]
  labels match?    : True


---
## 5  Visualisations

### 5.1  MNIST_seq1 — single sample as a time-series

MNIST_seq1 reads each 28×28 image pixel-by-pixel (1 pixel per step), producing a sequence of length 784.

In [35]:
X_seq, labels = load_data_as_sequence("MNIST_seq1", local=True, label_every_step=False)
info_mnist    = load_data("MNIST_seq1", local=True)["info"]
seq_len_mnist = info_mnist["seq_len"]

sample_indices = [0, 1, 2]
fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=[
        f"Sample {i}  |  label: {int(np.argmax(labels[i]))}"
        for i in sample_indices
    ],
)

for col, idx in enumerate(sample_indices, start=1):
    pixel_signal = X_seq[idx, :, 0]
    fig.add_trace(
        go.Scatter(
            y=pixel_signal,
            mode='lines',
            line=dict(color='steelblue', width=0.8),
            name=f"sample {idx}",
            showlegend=False,
        ),
        row=1, col=col,
    )
    fig.update_xaxes(title_text="Timestep (pixel index)", row=1, col=col)
    fig.update_yaxes(title_text="Pixel value", row=1, col=col)

fig.update_layout(
    title="MNIST_seq1 — pixel-by-pixel time series",
    height=350,
)
fig.show()

### 5.2  MNIST_seq1 — original 2-D image reconstructed from the sequence

In [36]:
fig = make_subplots(
    rows=2, cols=5,
    subplot_titles=[f"digit: {int(np.argmax(labels[i]))}" for i in range(10)],
    horizontal_spacing=0.04,
    vertical_spacing=0.08,
)

for idx in range(10):
    img   = X_seq[idx, :, 0].reshape(28, 28)
    row   = idx // 5 + 1
    col   = idx % 5  + 1
    fig.add_trace(
        go.Heatmap(
            z=img,
            colorscale='gray',
            showscale=False,
            hovertemplate="row %{y}  col %{x}<br>value: %{z:.3f}<extra></extra>",
        ),
        row=row, col=col,
    )
    fig.update_xaxes(showticklabels=False, row=row, col=col)
    fig.update_yaxes(showticklabels=False, autorange='reversed', row=row, col=col)

fig.update_layout(
    title="MNIST_seq1 — images reconstructed from X_seq[:, :, 0].reshape(28, 28)",
    height=420,
)
fig.show()

### 5.3  MNIST1D_seq1 — 1-D signal time series

In [37]:
X_1d, labels_1d = load_data_as_sequence("MNIST1D_seq1", local=True, label_every_step=False)
info_1d          = load_data("MNIST1D_seq1", local=True)["info"]

print(f"MNIST1D_seq1 info: {info_1d}")
print(f"X_seq shape: {X_1d.shape}")

sample_indices_1d = [0, 1, 2]
fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=[
        f"Sample {i}  |  label: {int(np.argmax(labels_1d[i]))}"
        for i in sample_indices_1d
    ],
)

for col, idx in enumerate(sample_indices_1d, start=1):
    signal = X_1d[idx, :, 0]
    fig.add_trace(
        go.Scatter(
            y=signal,
            mode='lines+markers',
            marker=dict(size=5),
            line=dict(color='darkorange', width=1.2),
            showlegend=False,
        ),
        row=1, col=col,
    )
    fig.update_xaxes(title_text="Timestep", row=1, col=col)
    fig.update_yaxes(title_text="Value", row=1, col=col)

fig.update_layout(
    title="MNIST1D_seq1 — 1-D template signals",
    height=350,
)
fig.show()

MNIST1D_seq1 info: {'num_points': 1000, 'size': 50, 'x_y_index': 40, 'x_size': 40, 'y_size': 10, 'onehot_y': 1, 'seq_len': 40, 'step_size': 1, 'data_family': 'MNIST1D'}
X_seq shape: (1000, 40, 1)


### 5.4  Sequence heatmap — feature × time

Visualising the full `(seq_len, channels)` slice of a single sample as a heatmap.  
This is most interesting when `step_size > 1` (e.g. one row of the image per timestep).

In [38]:
def plot_sequence_heatmap(X_seq, labels, sample_idx=0, title=""):
    """Plot a single sample's sequence as a heatmap (feature channel × timestep)."""
    seq   = X_seq[sample_idx]            # (seq_len, channels)
    label = int(np.argmax(labels[sample_idx]))

    fig = go.Figure(
        go.Heatmap(
            z=seq.T,                     # (channels, timesteps)
            colorscale='RdBu',
            zmid=0,
            colorbar=dict(title="Value"),
            hovertemplate="timestep %{x}<br>channel %{y}<br>value %{z:.4f}<extra></extra>",
        )
    )
    fig.update_layout(
        title=f"{title}  —  sample {sample_idx}, label {label}",
        xaxis_title="Timestep",
        yaxis_title="Feature channel",
        height=300,
    )
    fig.show()


# MNIST_seq1 with label embedded (step_size=1, label_dim=10 → 11 channels total)
X_embed, labels_embed = load_data_as_sequence("MNIST_seq1", local=True, label_every_step=True)
plot_sequence_heatmap(X_embed, labels_embed, sample_idx=0,
                      title="MNIST_seq1  label_every_step=True")

### 5.5  Label distribution in the loaded dataset

In [39]:
def plot_label_distribution(labels, dataset_name):
    label_ids = np.argmax(labels, axis=1)
    unique, counts = np.unique(label_ids, return_counts=True)

    fig = go.Figure(
        go.Bar(
            x=unique,
            y=counts,
            marker_color='steelblue',
            text=counts,
            textposition='outside',
        )
    )
    fig.update_layout(
        title=f"Label distribution — {dataset_name}",
        xaxis=dict(title="Class (argmax of one-hot label)", tickmode='linear', dtick=1),
        yaxis_title="Count",
        height=300,
    )
    fig.show()


plot_label_distribution(labels,    "MNIST_seq1")
plot_label_distribution(labels_1d, "MNIST1D_seq1")

---
## 6  Error handling — non-sequence datasets

`load_data_as_sequence` raises a descriptive `ValueError` when called on a dataset that has no sequence metadata.

In [40]:
try:
    load_data_as_sequence("MNIST", local=True)
except ValueError as e:
    print(f"ValueError: {e}")

ValueError: Dataset 'MNIST' has no sequence metadata (seq_len / step_size). Use load_data_as_xy() instead, or load a sequence variant such as 'MNIST_seq1'.


---
## 7  Interactive Dataset Explorer — Sequence Builder

This explorer lets you:

- **Choose any dataset** that has a pixel/label split (not just pre-built `_seq` variants)
- **Select `pixels_per_step`** — any divisor of the total pixel count — which determines how many pixels form one timestep and therefore the total sequence length
- **Build the sequence entry by entry** using the *Up to step* slider — drag it from 1 to `seq_len` to watch the sequence (and image) grow

Three panels update in sync:

| Panel | Content |
|---|---|
| **Left** | Revealed pixel signal (all pixels decoded so far, as a 1-D trace) |
| **Centre** | Heatmap of revealed timesteps × pixels-per-step |
| **Right** | For MNIST: 28×28 image with unrevealed pixels masked (NaN → white); for other datasets: one-hot label bar |

Changing `pixels_per_step` re-segments the same flat pixel array — no re-download needed.

In [41]:
# ---------------------------------------------------------------------------
# Raw-data cache — load flat pixel arrays once per dataset
# ---------------------------------------------------------------------------
_raw_cache = {}

def get_raw_data(base_name):
    """Return (pixels, labels, info) for base_name.

    pixels : (num_points, total_pixels)  float32
    labels : (num_points, label_dim)     float32
    info   : dict from info.json
    """
    if base_name not in _raw_cache:
        data  = load_data(base_name, local=True)
        info  = data["info"]
        tgt   = data["target"]
        idx   = info["x_y_index"]
        pixels = tgt.iloc[:, :idx].to_numpy(dtype=np.float32)
        labels = tgt.iloc[:, idx:].to_numpy(dtype=np.float32)
        _raw_cache[base_name] = (pixels, labels, info)
    return _raw_cache[base_name]


def divisors(n):
    """Return all positive divisors of n in ascending order."""
    return [d for d in range(1, n + 1) if n % d == 0]


# Detect datasets that have a pixel/label split (x_y_index present)
_base_names = []
for _n in sorted(data_names(local=True)):
    try:
        _inf = load_data(_n, local=True)["info"]
        if "x_y_index" in _inf:
            _base_names.append(_n)
    except Exception:
        pass

print("Datasets available in the sequence builder:")
for _n in _base_names:
    _pix, _, _inf = get_raw_data(_n)
    print(f"  {_n:50s}  total_pixels={_pix.shape[1]}  label_dim={_inf['y_size']}")


# ---------------------------------------------------------------------------
# FigureWidgets — created once, mutated in-place
# ---------------------------------------------------------------------------
fw_sig = go.FigureWidget(layout=go.Layout(
    title="Revealed pixel signal",
    xaxis_title="Pixel index",
    yaxis_title="Value",
    height=340,
    margin=dict(t=55, b=50, l=55, r=20),
))
fw_sig.add_scatter(mode='lines', line=dict(width=0.8, color='steelblue'), name='signal')

fw_map = go.FigureWidget(layout=go.Layout(
    title="Sequence heatmap",
    xaxis_title="Timestep",
    yaxis_title="Pixel within step",
    height=340,
    margin=dict(t=55, b=50, l=60, r=20),
))
fw_map.add_heatmap(colorscale='RdBu', zmid=0, showscale=True)

fw_img = go.FigureWidget(layout=go.Layout(
    title="Revealed image / label",
    height=340,
    margin=dict(t=55, b=50, l=50, r=20),
))
fw_img.add_heatmap(colorscale='gray', showscale=False)


# ---------------------------------------------------------------------------
# Update function
# ---------------------------------------------------------------------------
def update_builder(base_name, sample_idx, pixels_per_step, up_to_step, show_image):
    pixels, labels, info = get_raw_data(base_name)
    total_pixels = pixels.shape[1]
    label_dim    = info["y_size"]
    n_points     = info["num_points"]

    # Guard: pixels_per_step must divide total_pixels
    if total_pixels % pixels_per_step != 0:
        valid = [d for d in divisors(total_pixels) if d <= pixels_per_step]
        pixels_per_step = valid[-1] if valid else 1

    seq_len    = total_pixels // pixels_per_step
    up_to_step = max(1, min(up_to_step, seq_len))

    sample_idx  = min(sample_idx, n_points - 1)
    pixel_flat  = pixels[sample_idx]         # (total_pixels,)
    lbl         = labels[sample_idx]         # (label_dim,)
    digit       = int(np.argmax(lbl))

    # Reshape the full flat array into (seq_len, pixels_per_step)
    seq_full   = pixel_flat.reshape(seq_len, pixels_per_step)

    # Partial: only the first up_to_step timesteps
    n_revealed  = up_to_step * pixels_per_step
    partial_seq = seq_full[:up_to_step]      # (up_to_step, pixels_per_step)
    pct         = int(100 * up_to_step / seq_len)

    # ── left: revealed pixel signal ──────────────────────────────────────────
    with fw_sig.batch_update():
        fw_sig.data[0].x = list(range(n_revealed))
        fw_sig.data[0].y = pixel_flat[:n_revealed].tolist()
        fw_sig.data[0].mode = 'lines' if n_revealed > 80 else 'lines+markers'
        fw_sig.layout.title.text = (
            f"{base_name}  sample {sample_idx}  |  "
            f"{n_revealed}/{total_pixels} pixels revealed ({pct}%)  |  label: {digit}"
        )

    # ── centre: heatmap of revealed timesteps ────────────────────────────────
    with fw_map.batch_update():
        fw_map.data[0].z = partial_seq.T.tolist() if up_to_step > 0 else [[]]
        fw_map.layout.title.text = (
            f"Step {up_to_step}/{seq_len}  "
            f"({pixels_per_step} px/step  →  seq_len={seq_len})"
        )
        fw_map.layout.xaxis.title.text = f"Timestep (0–{up_to_step - 1})"
        fw_map.layout.yaxis.title.text = f"Pixel within step (0–{pixels_per_step - 1})"

    # ── right: image (MNIST) or label bar ────────────────────────────────────
    is_mnist = (total_pixels == 784 and show_image)
    if is_mnist:
        # Build image with revealed pixels; unrevealed → NaN (renders white)
        img_flat = np.full(total_pixels, np.nan, dtype=np.float32)
        img_flat[:n_revealed] = pixel_flat[:n_revealed]
        img = img_flat.reshape(28, 28)

        if not isinstance(fw_img.data[0], go.Heatmap):
            fw_img.data = []
            fw_img.add_heatmap(colorscale='gray', showscale=False,
                               hovertemplate="row %{y}  col %{x}<br>val %{z:.3f}<extra></extra>")
        with fw_img.batch_update():
            fw_img.data[0].z = img.tolist()
            fw_img.layout.title.text = f"28×28 image — step {up_to_step}/{seq_len}  |  digit: {digit}"
            fw_img.layout.yaxis.autorange  = 'reversed'
            fw_img.layout.xaxis.showticklabels = False
            fw_img.layout.yaxis.showticklabels = False
    else:
        bar_colors = ['tomato' if i == digit else 'steelblue' for i in range(label_dim)]
        if not isinstance(fw_img.data[0], go.Bar):
            fw_img.data = []
            fw_img.add_bar()
        with fw_img.batch_update():
            fw_img.data[0].x = list(range(label_dim))
            fw_img.data[0].y = lbl.tolist()
            fw_img.data[0].marker.color = bar_colors
            fw_img.layout.title.text  = f"Label vector  |  digit: {digit}"
            fw_img.layout.xaxis.title.text = "Class"
            fw_img.layout.yaxis.title.text = "One-hot value"
            fw_img.layout.yaxis.autorange  = True
            fw_img.layout.xaxis.showticklabels = True


# ---------------------------------------------------------------------------
# Widgets
# ---------------------------------------------------------------------------
_init_name   = _base_names[0] if _base_names else "MNIST"
_init_pixels, _, _init_info = get_raw_data(_init_name)
_init_total  = _init_pixels.shape[1]
_init_divs   = divisors(_init_total)
_init_seqlen = _init_total  # start with pixels_per_step=1

wb_dataset = widgets.Dropdown(
    options=_base_names,
    value=_init_name,
    description='Dataset:',
    layout=widgets.Layout(width='300px'),
)

wb_sample = widgets.IntSlider(
    value=0, min=0, max=_init_info['num_points'] - 1, step=1,
    description='Sample:',
    continuous_update=False,
    layout=widgets.Layout(width='380px'),
    style={'description_width': '70px'},
)

wb_pps = widgets.SelectionSlider(
    options=_init_divs,
    value=1,
    description='px / step:',
    continuous_update=False,
    readout=True,
    layout=widgets.Layout(width='380px'),
    style={'description_width': '80px'},
)

wb_step = widgets.IntSlider(
    value=_init_seqlen, min=1, max=_init_seqlen, step=1,
    description='Up to step:',
    continuous_update=False,
    layout=widgets.Layout(width='380px'),
    style={'description_width': '80px'},
)

wb_show_img = widgets.Checkbox(
    value=True,
    description='Show 28×28 image (MNIST)',
)


def _on_wb_dataset_change(change):
    pix, _, inf = get_raw_data(change['new'])
    total = pix.shape[1]
    divs  = divisors(total)
    wb_pps.options  = divs
    wb_pps.value    = 1
    wb_sample.max   = inf['num_points'] - 1
    wb_sample.value = 0
    wb_step.max     = total
    wb_step.value   = total


def _on_wb_pps_change(change):
    pix, _, _ = get_raw_data(wb_dataset.value)
    total     = pix.shape[1]
    pps       = change['new']
    seq_len   = total // pps
    # Preserve revealed fraction
    frac = wb_step.value / wb_step.max if wb_step.max > 0 else 1.0
    wb_step.max   = seq_len
    wb_step.value = max(1, min(round(frac * seq_len), seq_len))


wb_dataset.observe(_on_wb_dataset_change, names='value')
wb_pps.observe(_on_wb_pps_change, names='value')


# ── layout ───────────────────────────────────────────────────────────────────
ui_builder = widgets.VBox([
    widgets.HTML(
        "<b>Sequence Builder</b> — drag <i>Up to step</i> to reveal the sequence "
        "one timestep at a time; change <i>px/step</i> to re-segment."
    ),
    widgets.HBox([wb_dataset, wb_sample]),
    widgets.HBox([wb_pps,    wb_step]),
    widgets.HBox([wb_show_img]),
])

out_builder = widgets.interactive_output(
    update_builder,
    dict(
        base_name       = wb_dataset,
        sample_idx      = wb_sample,
        pixels_per_step = wb_pps,
        up_to_step      = wb_step,
        show_image      = wb_show_img,
    ),
)

panels_builder = widgets.HBox(
    [fw_sig, fw_map, fw_img],
    layout=widgets.Layout(width='100%'),
)

display(ui_builder, panels_builder, out_builder)

# Initial render
update_builder(
    wb_dataset.value, wb_sample.value,
    wb_pps.value, wb_step.value, wb_show_img.value,
)


Datasets available in the sequence builder:
  EMlocalization                                      total_pixels=160  label_dim=1
  LunarLander                                         total_pixels=404  label_dim=4
  MNIST                                               total_pixels=784  label_dim=10
  MNIST1D                                             total_pixels=40  label_dim=10
  MNIST1D_seq1                                        total_pixels=40  label_dim=10
  MNIST1Dcustom_scale0.4_maxtrans48_corrnoise0.25_iidnoise0.02_shear0.75  total_pixels=40  label_dim=10
  MNIST_custom_degrees0_0_translate0_0_scale1_1       total_pixels=2500  label_dim=10
  MNIST_seq1                                          total_pixels=784  label_dim=10
  MassSpec                                            total_pixels=921  label_dim=512
  regression_circle                                   total_pixels=1  label_dim=1
  regression_line                                     total_pixels=1  label_dim=1


    'data': [{'line': {'color': 'steelblue', 'width': 0.8},
              'mode'…

Output()

---
## 8  Training an LSTM on a Sequence Dataset

This section shows a complete, runnable training loop.

### How `pixels_per_step` shapes the model

The flat pixel array `(num_points, total_pixels)` is reshaped into
`(num_points, seq_len, pixels_per_step)` before being fed to the LSTM:

| `pixels_per_step` | `seq_len` (MNIST1D, 40 px) | `input_size` to LSTM |
|---|---|---|
| 1 | 40 | 1 |
| 2 | 20 | 2 |
| 4 | 10 | 4 |
| 5 |  8 | 5 |
| 8 |  5 | 8 |

A larger `pixels_per_step` means a *shorter* sequence with a *wider* input — the LSTM
sees more spatial context per step but fewer recurrent transitions.

The raw pixel data is loaded once and reshaped on-the-fly, so no new dataset files
are needed for each `pixels_per_step` value.


In [42]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader, random_split

# ── Experiment config ────────────────────────────────────────────────────────
DATASET         = "MNIST1D"   # any base dataset loaded in section 7
PIXELS_PER_STEP = 2           # must be a divisor of total_pixels
HIDDEN_SIZE     = 64
NUM_LAYERS      = 1
NUM_EPOCHS      = 20
BATCH_SIZE      = 64
LEARNING_RATE   = 1e-3
VAL_FRAC        = 0.2
DEVICE          = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Device : {DEVICE}")
print(f"Config : dataset={DATASET}  pixels_per_step={PIXELS_PER_STEP}  "
      f"hidden={HIDDEN_SIZE}  layers={NUM_LAYERS}  epochs={NUM_EPOCHS}")


Device : cuda
Config : dataset=MNIST1D  pixels_per_step=2  hidden=64  layers=1  epochs=20


In [43]:
# ── Load flat pixel array and reshape ───────────────────────────────────────
pixels, labels, info = get_raw_data(DATASET)   # reuses section 7 cache
total_pixels = pixels.shape[1]
label_dim    = info["y_size"]

assert total_pixels % PIXELS_PER_STEP == 0, (
    f"PIXELS_PER_STEP={PIXELS_PER_STEP} must divide total_pixels={total_pixels}. "
    f"Valid values: {divisors(total_pixels)}"
)

seq_len = total_pixels // PIXELS_PER_STEP
X_seq   = pixels.reshape(-1, seq_len, PIXELS_PER_STEP)    # (N, T, C)
y_hard  = np.argmax(labels, axis=1).astype(np.int64)       # (N,)

print(f"Dataset        : {DATASET}")
print(f"total_pixels   : {total_pixels}")
print(f"pixels_per_step: {PIXELS_PER_STEP}")
print(f"seq_len        : {seq_len}")
print(f"X_seq shape    : {X_seq.shape}   (num_points, seq_len, pixels_per_step)")
print(f"y_hard shape   : {y_hard.shape}")

# ── Train / validation split ─────────────────────────────────────────────────
X_t = torch.tensor(X_seq,   dtype=torch.float32)
y_t = torch.tensor(y_hard,  dtype=torch.long)

full_ds          = TensorDataset(X_t, y_t)
n_val            = int(len(full_ds) * VAL_FRAC)
n_train          = len(full_ds) - n_val
train_ds, val_ds = random_split(
    full_ds, [n_train, n_val],
    generator=torch.Generator().manual_seed(42),
)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE)

print(f"\nTrain samples  : {n_train}")
print(f"Val   samples  : {n_val}")
print(f"Batches/epoch  : {len(train_loader)}")


Dataset        : MNIST1D
total_pixels   : 40
pixels_per_step: 2
seq_len        : 20
X_seq shape    : (1000, 20, 2)   (num_points, seq_len, pixels_per_step)
y_hard shape   : (1000,)

Train samples  : 800
Val   samples  : 200
Batches/epoch  : 13


In [44]:
# ── Model definition ─────────────────────────────────────────────────────────
class LSTMClassifier(nn.Module):
    """Single- or multi-layer LSTM followed by a linear classification head.

    Args:
        input_size  : pixels_per_step (number of features per timestep)
        hidden_size : LSTM hidden dimension
        num_layers  : number of stacked LSTM layers
        num_classes : number of output classes
    """
    def __init__(self, input_size, hidden_size, num_layers, num_classes):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size  = input_size,
            hidden_size = hidden_size,
            num_layers  = num_layers,
            batch_first = True,
            dropout     = 0.2 if num_layers > 1 else 0.0,
        )
        self.head = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        # x      : (batch, seq_len, input_size)
        # h_n    : (num_layers, batch, hidden_size)
        _, (h_n, _) = self.lstm(x)
        return self.head(h_n[-1])   # classify from last layer's final hidden state


model     = LSTMClassifier(
    input_size  = PIXELS_PER_STEP,
    hidden_size = HIDDEN_SIZE,
    num_layers  = NUM_LAYERS,
    num_classes = label_dim,
).to(DEVICE)

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
criterion = nn.CrossEntropyLoss()

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters : {n_params:,}")
print(model)


Trainable parameters : 18,058
LSTMClassifier(
  (lstm): LSTM(2, 64, batch_first=True)
  (head): Linear(in_features=64, out_features=10, bias=True)
)


In [45]:
# ── Training loop ────────────────────────────────────────────────────────────
history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

for epoch in range(1, NUM_EPOCHS + 1):

    # --- train ---
    model.train()
    t_loss, t_correct, t_n = 0.0, 0, 0
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        logits = model(xb)
        loss   = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        t_loss    += loss.item() * len(yb)
        t_correct += (logits.argmax(1) == yb).sum().item()
        t_n       += len(yb)
    history["train_loss"].append(t_loss / t_n)
    history["train_acc"].append(t_correct / t_n)

    # --- validate ---
    model.eval()
    v_loss, v_correct, v_n = 0.0, 0, 0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            logits  = model(xb)
            v_loss    += criterion(logits, yb).item() * len(yb)
            v_correct += (logits.argmax(1) == yb).sum().item()
            v_n       += len(yb)
    history["val_loss"].append(v_loss / v_n)
    history["val_acc"].append(v_correct / v_n)

    print(f"Epoch {epoch:3d}/{NUM_EPOCHS}  "
          f"train loss {history['train_loss'][-1]:.4f}  acc {history['train_acc'][-1]:.3f}  |  "
          f"val loss {history['val_loss'][-1]:.4f}  acc {history['val_acc'][-1]:.3f}")

print("\nTraining complete.")
print(f"Best val accuracy : {max(history['val_acc']):.3f}  "
      f"(epoch {history['val_acc'].index(max(history['val_acc'])) + 1})")


Epoch   1/20  train loss 2.3038  acc 0.096  |  val loss 2.3046  acc 0.110
Epoch   2/20  train loss 2.2982  acc 0.124  |  val loss 2.3004  acc 0.110
Epoch   3/20  train loss 2.2908  acc 0.131  |  val loss 2.2868  acc 0.140
Epoch   4/20  train loss 2.2288  acc 0.181  |  val loss 2.1665  acc 0.230
Epoch   5/20  train loss 2.0854  acc 0.224  |  val loss 2.0686  acc 0.225
Epoch   6/20  train loss 1.9736  acc 0.233  |  val loss 2.0321  acc 0.205
Epoch   7/20  train loss 1.9157  acc 0.235  |  val loss 1.9572  acc 0.215
Epoch   8/20  train loss 1.8643  acc 0.260  |  val loss 1.8997  acc 0.230
Epoch   9/20  train loss 1.8181  acc 0.295  |  val loss 1.8421  acc 0.265
Epoch  10/20  train loss 1.7767  acc 0.310  |  val loss 1.8021  acc 0.280
Epoch  11/20  train loss 1.7502  acc 0.307  |  val loss 1.7775  acc 0.280
Epoch  12/20  train loss 1.7432  acc 0.329  |  val loss 1.7904  acc 0.300
Epoch  13/20  train loss 1.7196  acc 0.333  |  val loss 1.8044  acc 0.280
Epoch  14/20  train loss 1.7015  acc 0

In [46]:
# ── Training curves ──────────────────────────────────────────────────────────
epochs = list(range(1, NUM_EPOCHS + 1))

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=["Cross-entropy loss", "Accuracy"],
)

for split, color in [("train", "steelblue"), ("val", "tomato")]:
    fig.add_trace(
        go.Scatter(
            x=epochs, y=history[f"{split}_loss"],
            name=f"{split} loss",
            line=dict(color=color, width=2),
            legendgroup=split,
            hovertemplate=f"%{{x}}  {split} loss=%{{y:.4f}}<extra></extra>",
        ),
        row=1, col=1,
    )
    fig.add_trace(
        go.Scatter(
            x=epochs, y=history[f"{split}_acc"],
            name=f"{split} acc",
            line=dict(color=color, width=2, dash="dash"),
            legendgroup=split,
            hovertemplate=f"%{{x}}  {split} acc=%{{y:.3f}}<extra></extra>",
        ),
        row=1, col=2,
    )

fig.update_xaxes(title_text="Epoch")
fig.update_yaxes(title_text="Loss",     row=1, col=1)
fig.update_yaxes(title_text="Accuracy", row=1, col=2, range=[0, 1])
fig.update_layout(
    title=(
        f"LSTM — {DATASET}  |  "
        f"pixels_per_step={PIXELS_PER_STEP}  seq_len={seq_len}  "
        f"hidden={HIDDEN_SIZE}  layers={NUM_LAYERS}"
    ),
    height=400,
    legend=dict(groupclick="toggleitem"),
)
fig.show()

# ── Confusion matrix ─────────────────────────────────────────────────────────
model.eval()
all_pred, all_true = [], []
with torch.no_grad():
    for xb, yb in val_loader:
        pred = model(xb.to(DEVICE)).argmax(1).cpu()
        all_pred.append(pred)
        all_true.append(yb)

all_pred = torch.cat(all_pred).numpy()
all_true = torch.cat(all_true).numpy()

num_classes = label_dim
conf = np.zeros((num_classes, num_classes), dtype=int)
for t, p in zip(all_true, all_pred):
    conf[t, p] += 1

fig_cm = go.Figure(
    go.Heatmap(
        z=conf,
        x=list(range(num_classes)),
        y=list(range(num_classes)),
        colorscale="Blues",
        text=conf,
        texttemplate="%{text}",
        hovertemplate="true %{y}  pred %{x}<br>count %{z}<extra></extra>",
    )
)
fig_cm.update_layout(
    title=f"Confusion matrix (validation set)  |  val acc={max(history['val_acc']):.3f}",
    xaxis_title="Predicted class",
    yaxis_title="True class",
    yaxis_autorange="reversed",
    height=420,
    width=480,
)
fig_cm.show()
